In [1]:
%pip install chromadb
%pip install sentence-transformers

import chromadb
from sentence_transformers import SentenceTransformer

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# ==========================================
# STEP 1: Initialize Database Client
# ==========================================
# Instantiate an ephemeral, in-memory ChromaDB client (ideal for testing and rapid prototyping)
chroma_client = chromadb.Client()

In [3]:
# ==========================================
# STEP 2: Configure & Create Collections
# ==========================================
# Create distinct collections for testing three different pre-trained embedding models.
# Note: ChromaDB defaults to L2 (Euclidean) distance unless configured otherwise.
collection_model1 = chroma_client.create_collection(name="all-MiniLM-L6-v2")
collection_model2 = chroma_client.create_collection(name="all-mpnet-base-v2")
collection_model3 = chroma_client.create_collection(name="BAAI-bge-small-en-v1.5")

In [4]:
# ==========================================
# STEP 3: Define Toy Dataset & Metadata
# ==========================================
# A small document corpus representing different concepts to evaluate embedding accuracy
documents = [
    "The quick brown fox jumps over the lazy dog.",
    "Artificial Intelligence and Machine Learning are transforming industries.",
    "A beautiful sunset over the calm ocean waves.",
]

# Unique string identifiers for each document entry
ids = ["doc1", "doc2", "doc3"]

# Optional rich metadata to attach for context, advanced filtering, or payload discovery
metadatas = [
    {"category": "animals", "length": "short"},
    {"category": "tech", "length": "medium"},
    {"category": "nature", "length": "short"},
]

In [5]:
# ==========================================
# STEP 4: Initialize Pre-trained Transformer Models
# ==========================================
# Load 3 distinct sentence-transformer models to see how their vector alignments differ
model1 = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
model2 = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
model3 = SentenceTransformer("BAAI/bge-small-en-v1.5")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [6]:
# ==========================================
# STEP 5: Define Search Query & Generate Vector Embeddings
# ==========================================
# The target conceptual query we want to match against our dataset
query = "A high tech industry innovation"

# Convert both documents and the search query into high-dimensional numerical vectors
# Model 1 Embeddings (384 Dimensions)
embeddings1 = model1.encode(documents).tolist()
query_emb1 = model1.encode(query).tolist()

# Model 2 Embeddings (768 Dimensions)
embeddings2 = model2.encode(documents).tolist()
query_emb2 = model2.encode(query).tolist()

# Model 3 Embeddings (384 Dimensions)
embeddings3 = model3.encode(documents).tolist()
query_emb3 = model3.encode(query).tolist()

In [7]:
# ==========================================
# STEP 6: Populate Vector Collections
# ==========================================
# Manually upsert documents along with their corresponding pre-computed embeddings
collection_model1.add(embeddings=embeddings1, documents=documents, metadatas=metadatas, ids=ids)
collection_model2.add(embeddings=embeddings2, documents=documents, metadatas=metadatas, ids=ids)
collection_model3.add(embeddings=embeddings3, documents=documents, metadatas=metadatas, ids=ids)

In [8]:
# ==========================================
# STEP 7: Execute Similarity Queries
# ==========================================
# Query collection 1 using all-MiniLM-L6-v2 vectors
results1 = collection_model1.query(query_embeddings=[query_emb1], n_results=3)

# Query collection 2 using all-mpnet-base-v2 vectors
results2 = collection_model2.query(query_embeddings=[query_emb2], n_results=3)

# Query collection 3 using BAAI-bge-small-en-v1.5 vectors
results3 = collection_model3.query(query_embeddings=[query_emb3], n_results=3)

In [9]:
# ==========================================
# STEP 8: Inspect Model Metric Performance
# ==========================================
# Print raw L2 distance scores. 
# Smaller distance means higher semantic similarity (closer together in vector space).

print("all-MiniLM-L6-v2: \n")
print(results1["distances"])
print("-" * 40)

print("all-mpnet-base-v2: \n")
print(results2["distances"])
print("-" * 40)

print("BAAI/bge-small-en-v1.5: \n")
print(results3["distances"])

all-MiniLM-L6-v2: 

[[1.0142298936843872, 1.7705512046813965, 1.9446276426315308]]
----------------------------------------
all-mpnet-base-v2: 

[[1.1018568277359009, 1.8033318519592285, 1.9036905765533447]]
----------------------------------------
BAAI/bge-small-en-v1.5: 

[[0.4901418089866638, 1.1322288513183594, 1.2605805397033691]]


In [ ]:
# 📘 Notes: Semantic Search using ChromaDB

## 🎯 Objective

The goal of this notebook is to understand how a Vector Database works by storing document embeddings in ChromaDB and performing semantic search.

Instead of manually calculating cosine similarity, ChromaDB handles storage, indexing, and retrieval of embeddings.

---

## Why This Notebook Matters

Previous notebooks used:

```text
Documents
    ↓
Embeddings
    ↓
Cosine Similarity
    ↓
Results
```

This notebook introduces:

```text
Documents
    ↓
Embeddings
    ↓
Vector Database
    ↓
Similarity Search
    ↓
Results
```

This is the core architecture behind modern RAG systems.

---

## Overall Workflow

```text
Documents
    ↓
Embedding Models
    ↓
Vector Embeddings
    ↓
Store in ChromaDB
    ↓
Query Embedding
    ↓
Vector Search
    ↓
Most Similar Documents
```

---

## Step 1: Initialize ChromaDB

```python
chroma_client = chromadb.Client()
```

Creates an in-memory ChromaDB instance.

Current notebook uses:

```text
Ephemeral Database
```

Meaning:

```text
Notebook Stops
    ↓
Database Disappears
```

Useful for:

- Learning
- Testing
- Experiments

Not suitable for production.

---

## What is a Vector Database?

Traditional Database:

```text
ID | Name | Salary
```

Searches exact values.

Example:

```sql
SELECT * FROM employees
WHERE name = 'John'
```

---

Vector Database:

```text
Document
↓
Embedding
↓
Vector
```

Searches by meaning.

Example:

```text
Query:
"AI innovation"

Matches:

"Machine Learning is transforming industries"
```

even if the exact words are different.

---

## Step 2: Collections

Three collections are created.

```python
collection_model1
collection_model2
collection_model3
```

Each collection stores embeddings generated by a different model.

Think of a collection as:

```text
SQL Table
```

but designed for vectors.

---

## Why Multiple Collections?

We want to compare how different embedding models perform.

Models:

1. all-MiniLM-L6-v2
2. all-mpnet-base-v2
3. BAAI/bge-small-en-v1.5

---

## Step 3: Documents

Dataset contains three documents.

### Document 1

```text
The quick brown fox jumps over the lazy dog.
```

Category:

```text
Animals
```

---

### Document 2

```text
Artificial Intelligence and Machine Learning are transforming industries.
```

Category:

```text
Technology
```

---

### Document 3

```text
A beautiful sunset over the calm ocean waves.
```

Category:

```text
Nature
```

---

## Metadata

Additional information attached to documents.

Example:

```python
{
    "category": "tech",
    "length": "medium"
}
```

Metadata can later be used for:

- Filtering
- Search constraints
- Analytics

Example:

```text
Find only tech documents
```

---

## Step 4: Embedding Models

Three embedding models are compared.

---

### all-MiniLM-L6-v2

Dimension:

```text
384
```

Characteristics:

- Fast
- Lightweight
- Popular for RAG

---

### all-mpnet-base-v2

Dimension:

```text
768
```

Characteristics:

- Higher quality embeddings
- Better semantic understanding
- Slower than MiniLM

---

### BAAI/bge-small-en-v1.5

Dimension:

```text
384
```

Characteristics:

- Designed specifically for retrieval tasks
- Frequently used in production RAG systems

---

## Step 5: Embedding Generation

Documents:

```python
model.encode(documents)
```

Query:

```python
model.encode(query)
```

Example:

```text
"A high tech industry innovation"
```

becomes:

```python
[0.14, -0.91, 0.34, ...]
```

A dense numerical vector.

---

## Query Understanding

Query:

```text
A high tech industry innovation
```

Most semantically related document:

```text
Artificial Intelligence and Machine Learning are transforming industries.
```

Expected result:

```text
Document 2
```

should rank highest.

---

## Step 6: Insert into ChromaDB

```python
collection.add(...)
```

Stores:

- Documents
- Metadata
- IDs
- Embeddings

inside the vector database.

---

## Why Store Embeddings?

Without a vector database:

```text
Generate Embeddings
↓
Store in Python Lists
↓
Search Manually
```

With ChromaDB:

```text
Generate Embeddings
↓
Store in DB
↓
Search Efficiently
```

This becomes critical when handling:

- 10,000 documents
- 100,000 documents
- Millions of documents

---

## Step 7: Similarity Search

```python
collection.query(...)
```

Performs vector search.

Input:

```python
query_embedding
```

Output:

```python
Most Similar Documents
```

ChromaDB automatically:

1. Compares vectors
2. Computes distance
3. Ranks documents
4. Returns best matches

---

## Distance Scores

Output:

```python
results["distances"]
```

returns:

```text
Distance between vectors
```

Current collection uses:

```text
L2 Distance (Euclidean Distance)
```

---

## Understanding Distance

### Smaller Distance

```text
Query
↓
Document
```

Very similar.

Example:

```text
0.15
```

Good match.

---

### Larger Distance

```text
Query
↓
Document
```

Less similar.

Example:

```text
1.85
```

Weak match.

---

## Distance vs Similarity

Previous notebooks:

```text
Cosine Similarity
```

Higher score = Better

Example:

```text
0.92 > 0.65
```

---

Current notebook:

```text
L2 Distance
```

Lower score = Better

Example:

```text
0.15 < 1.20
```

---

## Why Compare Multiple Models?

Different models create different embeddings.

Same query:

```text
AI Innovation
```

may produce:

```text
Distance = 0.18
```

for one model and

```text
Distance = 0.45
```

for another.

Better models usually separate concepts more effectively.

---

## Relation to RAG

This notebook demonstrates the retrieval component of RAG.

Current Notebook:

```text
Query
↓
Embedding
↓
Vector Database
↓
Top Documents
```

RAG:

```text
User Question
↓
Embedding
↓
Vector Database
↓
Relevant Documents
↓
LLM
↓
Final Answer
```

The retrieval process is identical.

---

## Why ChromaDB?

Advantages:

- Open Source
- Easy Setup
- Python Friendly
- Works locally
- Popular for RAG prototypes

---

## Other Vector Databases

Popular alternatives:

- ChromaDB
- FAISS
- Pinecone
- Weaviate
- Milvus
- Qdrant

All solve the same problem:

```text
Store Embeddings
↓
Retrieve Similar Content
```

---

## Key Learnings

✅ Vector Databases

✅ ChromaDB Collections

✅ Embedding Storage

✅ Semantic Search

✅ Vector Retrieval

✅ Metadata Storage

✅ Distance Metrics

✅ Multi-Model Comparison

✅ Foundations of RAG

---

## Interview Questions

### What is a Vector Database?

A database optimized for storing and searching vector embeddings.

---

### Why use ChromaDB?

To efficiently store embeddings and perform semantic search.

---

### What is a Collection?

A logical grouping of documents and embeddings inside ChromaDB.

---

### What is the difference between Cosine Similarity and L2 Distance?

Cosine Similarity:

```text
Higher = Better
```

L2 Distance:

```text
Lower = Better
```

---

### Why store metadata?

To enable filtering and provide additional document context.

---

### How does this relate to RAG?

Vector databases retrieve relevant context before it is passed to an LLM for answer generation.

---

## 🚀 Big Picture

This notebook introduces the architecture used by most modern GenAI applications:

```text
Documents
    ↓
Embeddings
    ↓
Vector Database
    ↓
Semantic Search
    ↓
Relevant Context
    ↓
LLM
    ↓
Answer
```

Once this concept is understood, building RAG systems becomes much easier.